[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jseborga/recursos-ia/blob/main/notebooks/07_asistente_programacion.ipynb)

# Un asistente de programación gratuito dentro de Colab

Este cuaderno instala **Ollama** (un programa de código abierto para ejecutar modelos de lenguaje) en la máquina de Colab y descarga **Qwen2.5-Coder**, un modelo abierto especializado en programación (licencia Apache 2.0). Después le pedimos funciones de cálculo numérico y **Python las prueba antes de usarlas**: la IA programa, el auxiliar verifica.

| Modelo | Descarga (cuantizado a 4 bits) | Cuándo usarlo |
|---|---|---|
| `qwen2.5-coder:7b` | ≈ 4,7 GB | Con GPU T4 (recomendado) |
| `qwen2.5-coder:1.5b` | ≈ 1 GB | Sin GPU; más rápido, se equivoca más |

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*. La instalación y la descarga tardan de 2 a 4 minutos.

**Reglas de Colab gratuito:** el modelo se usa desde las celdas de este cuaderno. Las condiciones de Colab prohíben en los entornos gratuitos el control remoto (SSH, escritorio remoto) y usar Colab principalmente a través de una interfaz web externa, así que **no** se conecta a tu editor mediante túneles. Para tener un asistente dentro de tu editor, Ollama se instala en tu propia computadora.

**Otra opción ya integrada:** Colab tiene su propio asistente, Gemini (el botón de la estrella, abajo en la pantalla). Según la ayuda de Colab, requiere cuenta de 18 años o más y un país compatible.

In [ ]:
#@title 1 · Instalar Ollama y arrancar el servidor (≈ 1 minuto)
import subprocess, time, os, re, json, requests, torch

GPU = torch.cuda.is_available()
MODELO = "qwen2.5-coder:7b" if GPU else "qwen2.5-coder:1.5b"
print("GPU:", torch.cuda.get_device_name(0) if GPU else "no (CPU)", "· modelo:", MODELO)

def verificar(condicion, mensaje):
    if not condicion:
        raise AssertionError("✘ " + mensaje)
    print("✔", mensaje)

subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_instalacion.log 2>&1", shell=True)
servidor = subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:11434/api/version", timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
version = requests.get("http://127.0.0.1:11434/api/version", timeout=5).json()["version"]
verificar(bool(version), f"servidor Ollama {version} funcionando dentro de esta máquina de Colab")

In [ ]:
#@title 2 · Descargar el modelo (1 a 3 minutos)
t0 = time.time()
r = subprocess.run(["ollama", "pull", MODELO], capture_output=True, text=True)
modelos = [m["name"] for m in requests.get("http://127.0.0.1:11434/api/tags").json()["models"]]
print(f"Descargado en {time.time() - t0:.0f} s · modelos disponibles: {modelos}")
verificar(any(m.startswith(MODELO) for m in modelos), f"{MODELO} listo para usar")

def programar(pedido, temperatura=0.2):
    """Le pide código al modelo y devuelve el primer bloque de Python de la respuesta."""
    sistema = ("Eres un programador científico. Responde solo con un bloque de código Python completo y comentado en español, "
               "con los import necesarios, sin ejemplos de uso ni pruebas.")
    resp = requests.post("http://127.0.0.1:11434/api/chat", json={
        "model": MODELO, "stream": False,
        "messages": [{"role": "system", "content": sistema}, {"role": "user", "content": pedido}],
        "options": {"temperature": temperatura, "num_ctx": 4096, "seed": 2026}}, timeout=600).json()
    texto = resp["message"]["content"]
    bloques = re.findall(r"```(?:python)?\s*\n(.*?)```", texto, flags=re.S)
    return bloques[0] if bloques else texto

t0 = time.time()
print(programar("Una función cuadrado(x) que devuelva x al cuadrado."))
print(f"({time.time() - t0:.1f} s)")

## 3 · La IA programa, Python verifica

Cuatro pedidos típicos de un curso de métodos numéricos. Para cada uno: se muestra el código, **se ejecuta en un espacio aparte** y se somete a pruebas con respuesta conocida. Lean siempre el código antes de ejecutarlo; aquí son funciones de cálculo, sin acceso a archivos.

El cuarto pedido tiene una trampa conocida: `scipy.special.ellipk(m)` recibe el **parámetro** m = k², no el módulo k. Confundirlos da un período equivocado que «se ve razonable».

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

def periodo_numerico(L, g, th0):
    """Referencia independiente: integra el péndulo y mide dos cruces por cero."""
    cruce = lambda t, y: y[0]; cruce.direction = -1
    s = solve_ivp(lambda t, y: [y[1], -g/L*np.sin(y[0])], [0, 20], [th0, 0], events=cruce, rtol=1e-11, atol=1e-13)
    return s.t_events[0][1] - s.t_events[0][0]

def prueba_trapecio(ns):
    f = ns["trapecio"]
    e1 = abs(f(np.sin, 0, np.pi, 100) - 2); e2 = abs(f(np.sin, 0, np.pi, 200) - 2)
    return e1 < 2e-4 and 3.9 < e1/e2 < 4.1, f"error con n=100: {e1:.1e}; razón al duplicar n: {e1/e2:.2f} (debe ser ≈ 4)"

def prueba_rk4(ns):
    f = ns["rk4"]
    y = f(lambda t, y: -y, 0.0, np.array([1.0]), 1.0, 100)
    y = np.atleast_1d(np.asarray(y, dtype=float))[-1]
    err = abs(y - np.exp(-1))
    return err < 1e-8, f"y' = −y hasta t = 1: error {err:.1e} (RK4 con 100 pasos debe dar < 10⁻⁸)"

def prueba_biseccion(ns):
    f = ns["biseccion"]
    x = f(lambda x: x*x - 2, 0, 2, 1e-12)
    return abs(x - np.sqrt(2)) < 1e-10, f"raíz de x² − 2: {x!r} (error {abs(x - np.sqrt(2)):.1e})"

def prueba_pendulo(ns):
    f = ns["periodo_pendulo"]
    T = f(1.0, 9.78, np.radians(60)); ref = periodo_numerico(1.0, 9.78, np.radians(60))
    return abs(T - ref)/ref < 1e-6, f"θ₀ = 60°: modelo {T:.6f} s, integración numérica {ref:.6f} s"

PEDIDOS = [
    ("trapecio", "Escribe trapecio(f, a, b, n): la regla del trapecio compuesta con n subintervalos, usando numpy.", prueba_trapecio),
    ("rk4", "Escribe rk4(f, t0, y0, t1, n): integra y' = f(t, y) con Runge-Kutta de orden 4 clásico y n pasos iguales, "
            "donde y0 es un arreglo de numpy. Devuelve el valor de y en t1.", prueba_rk4),
    ("biseccion", "Escribe biseccion(f, a, b, tol): el método de bisección; lanza ValueError si f(a) y f(b) tienen el mismo signo.", prueba_biseccion),
    ("periodo_pendulo", "Escribe periodo_pendulo(L, g, theta0): el período EXACTO de un péndulo simple de amplitud theta0 (en radianes), "
                        "usando la integral elíptica completa de primera especie de scipy.special.", prueba_pendulo),
]

resultados = []
for nombre, pedido, prueba in PEDIDOS:
    codigo = programar(pedido)
    print(f"\n━━ {nombre} ━━\n{codigo}")
    ns = {}
    try:
        exec(codigo, ns)
        ok, detalle = prueba(ns)
    except Exception as ex:
        ok, detalle = False, f"falló al ejecutarse: {ex.__class__.__name__}: {ex}"
    print(("✔ " if ok else "✘ ") + detalle)
    resultados.append((nombre, ok))
print(f"\n{MODELO}: {sum(o for _, o in resultados)} de {len(resultados)} funciones pasaron sus pruebas.")

**Para el aula:** si una función falla, copien el mensaje del `✘` y pídanle al modelo que la corrija («la prueba dio este error: …»). Ese ciclo de pedir, probar y devolver el error es exactamente cómo se trabaja con cualquier asistente de programación, gratuito o de pago.

In [ ]:
#@title 4 · Tu propio pedido (el código se muestra, no se ejecuta) { display-mode: "form" }
PEDIDO = "Escribe una función que integre el oscilador armónico con el método de Verlet de velocidades y devuelva posiciones y energías."  #@param {type:"string"}
print(programar(PEDIDO))

## Qué muestra este cuaderno

- Un asistente de programación abierto puede correr gratis dentro de Colab, sin cuentas ni claves.
- El código generado **se prueba antes de usarse**: integrales con resultado exacto, órdenes de convergencia y una referencia numérica independiente.
- Para trabajar dentro de tu editor, el camino permitido es instalar Ollama en tu computadora; dentro de Colab, el asistente vive en las celdas.

Referencias: [Ollama](https://github.com/ollama/ollama) (licencia MIT) · [Qwen2.5-Coder-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct) (Apache 2.0) · [Preguntas frecuentes de Colab](https://research.google.com/colaboratory/faq.html) (usos permitidos).